# DATT — Chạy project trên Colab T4

Chạy lần lượt **Cell 1 → 13** trên runtime hiện tại. Không reset runtime. Source và model hiện có trong `/content/DATT`; bộ Torch CUDA 12.6 đã được kiểm chứng.

Cell 1 provisions Torch cu126 on fresh cu130 runtimes and reuses matching packages. If native modules are already loaded, restart the session as instructed, then rerun Cell 1; do not delete the runtime.

1. **Cell 1–3:** kiểm tra phép tính GPU, source và dependencies.
2. **Cell 4:** tự đọc `/content/DATT/.env` hoặc Colab Secrets. Nếu thiếu, nhập cấu hình qua ô nhập ẩn. Không dán mật khẩu vào code. Cần `DATT_DATABASE_URL`, `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`, `DATT_STORAGE_BUCKET`; backend mặc định `supabase`. SMTP dùng Secrets có sẵn hoặc ô nhập ẩn. Nếu dùng `.env`, điền đúng hoặc bỏ các dòng giá trị rỗng trước khi chạy.
3. **Cell 5–10:** kiểm tra PostgreSQL/pgvector/Storage, model, áp dụng migration hiện có và khởi động dịch vụ. Storage preflight tạo rồi xóa object thử riêng.
4. **Cell 11:** mở link ứng dụng được in ra. **Cell 12–13:** kiểm tra API và trạng thái startup.

Nếu cell lỗi: sửa nguyên nhân, chạy lại từ **Cell 1**. Không chạy vượt cell lỗi. Notebook dừng khi thiếu cấu hình, source/model hoặc dependency không tương thích. Runtime mới cần nạp lại source/model; notebook này dùng source đang triển khai, không tự clone repository.

`SYSTEM_READY=YES` chỉ xác nhận startup/smoke check thành công, chưa bao gồm Real Camera Acceptance. Cấu hình nhập ẩn chỉ lưu trong môi trường runtime; không ghi vào notebook hay `.env`.


In [ ]:
# CELL 1 — Provision the supported CUDA stack, then test the GPU
import os, sys, json, subprocess, socket
from pathlib import Path

ROOT = Path('/content/DATT')
startup = None
runtime = None
print('python=' + sys.version.split()[0])
expected = {'torch': '2.11.0+cu126', 'torchvision': '0.26.0+cu126', 'torchaudio': '2.11.0+cu126'}
version_code = "import importlib.metadata as m,json; names=('torch','torchvision','torchaudio'); print(json.dumps({n:m.version(n) for n in names if m.packages_distributions().get(n)}))"
versions = subprocess.run([sys.executable, '-c', version_code], capture_output=True, text=True, timeout=90)
if versions.returncode:
    raise RuntimeError('Khong doc duoc phien ban Torch trong runtime.')
if json.loads(versions.stdout) != expected:
    # Do not replace native libraries underneath a running kernel/service.
    if any(n in sys.modules for n in ('torch', 'torchvision', 'torchaudio', 'onnxruntime')):
        raise RuntimeError('Chon Runtime > Restart session, sau do chay lai Cell 1 de cai Torch cu126. Khong Delete runtime.')
    for port in (8000, 8501):
        with socket.socket() as sock:
            sock.settimeout(1)
            if sock.connect_ex(('127.0.0.1', port)) == 0:
                raise RuntimeError('Dung dich vu DATT bang CLI truoc khi doi Torch: port ' + str(port))
    if subprocess.run(['nvidia-smi', '-L'], capture_output=True, timeout=30).returncode:
        raise RuntimeError('Chon GPU T4 trong Runtime > Change runtime type.')
    print('Dang cai Torch CUDA 12.6 tu download.pytorch.org; co the mat vai phut.', flush=True)
    log_path = Path('/content/datt-torch-install.log')
    with log_path.open('w') as log:
        result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
            '--only-binary=:all:', '--index-url', 'https://download.pytorch.org/whl/cu126',
            *[n + '==' + v for n, v in expected.items()]], stdout=log, stderr=subprocess.STDOUT, timeout=1200)
    if result.returncode:
        raise RuntimeError('Cai Torch cu126 that bai; xem /content/datt-torch-install.log, sau do chay lai Cell 1.')
probe_code = "import torch,torchvision,torchaudio,json; x=torch.arange(4,device='cuda'); torch.cuda.synchronize(); print(json.dumps(dict(gpu=torch.cuda.get_device_name(0),cuda=torch.version.cuda,available=torch.cuda.is_available(),torch=torch.__version__,torchvision=torchvision.__version__,torchaudio=torchaudio.__version__,tensor_ok=(x.sum().item()==6))))"
probe = subprocess.run([sys.executable, '-c', probe_code], capture_output=True, text=True, timeout=90)
if probe.returncode:
    Path('/content/datt-gpu-probe.log').write_text(probe.stderr)
    raise RuntimeError('GPU smoke test FAIL; xem /content/datt-gpu-probe.log.')
info = json.loads(probe.stdout)
for key in ('gpu', 'torch', 'cuda', 'available', 'tensor_ok'):
    print(str(key) + '=' + str(info[key]))
if not info['available'] or info['cuda'] != '12.6' or not info['tensor_ok'] or any(info[n] != v for n, v in expected.items()):
    raise RuntimeError('Can bo Torch cu126 dong bo va GPU tensor test PASS de tiep tuc.')
runtime = dict(runtime='colab', gpu=info['gpu'], cuda=info['cuda'], runtime_pid=os.getpid())
print('cell1=PASS')


In [ ]:
# CELL 2 — Tai su dung source /content/DATT
if not isinstance(runtime, dict) or runtime.get('runtime') != 'colab':
    raise RuntimeError('Chay Cell 1 thanh cong truoc.')
for name in ('scripts/datt.py', 'scripts/colab_startup.py', 'scripts/colab_install.py', 'src/main.py', 'src/db/alembic.ini'):
    if not (ROOT / name).is_file():
        raise RuntimeError('Thieu source /content/DATT: ' + name + '. Hay nap lai source project truoc khi tiep tuc.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.colab_startup import Startup
startup = Startup(runtime)
print('source=PASS; root=' + str(ROOT))
print('commit=' + startup.report['commit'])


In [ ]:
# CELL 3 — Kiem tra va cai dependencies con thieu
# Giu nguyen bo Torch cu126 da kiem chung; dung neu phat hien xung dot.
startup.run(3)


In [ ]:
# CELL 4 — Cau hinh bao mat (khong dan mat khau vao code)
# Tu dong doc .env / Colab Secrets; chi hoi nhung gia tri con thieu.
import contextlib, io, getpass
from dotenv import load_dotenv
from scripts.colab_startup import REQUIRED, EMAIL
if startup is None or startup.failed or startup.completed != 3:
    raise RuntimeError('Chay lai Cell 1, 2, 3 thanh cong truoc Cell 4.')
with contextlib.redirect_stderr(io.StringIO()):
    load_dotenv(ROOT / '.env', override=True, interpolate=False)
from google.colab import userdata
for key in REQUIRED + EMAIL:
    if not os.getenv(key):
        try:
            secret_value = userdata.get(key)
            if secret_value:
                os.environ[key] = secret_value
            del secret_value
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
os.environ.setdefault('DATT_STORAGE_BACKEND', 'supabase')
print('Nhap cau hinh cho runtime Colab. Gia tri duoc an va khong ghi vao notebook.')
for key in REQUIRED:
    if not os.getenv(key):
        os.environ[key] = getpass.getpass(key + ': ').strip()
# SMTP da co trong Secrets se duoc tai su dung. De trong neu chua cau hinh.
for key in ('DATT_EMAIL_USERNAME', 'DATT_EMAIL_PASSWORD', 'DATT_EMAIL_FROM', 'DATT_EMAIL_TO'):
    if not os.getenv(key):
        os.environ[key] = getpass.getpass(key + ' (Enter de bo qua): ').strip()
startup.run(4)


In [ ]:
# CELL 5 — PostgreSQL / pgvector / Supabase Storage
# Kiem tra ket noi va thu upload/read/delete mot object tam rieng.
startup.run(5)


In [ ]:
# CELL 6 — Kiem tra model va GPU inference
# Tai su dung model hien co; kiem tra YOLO, SCRFD, AdaFace, OCR.
startup.run(6)


In [ ]:
# CELL 7 — Chay migration hien co
# Ap dung migration cua repository; khong reset database.
startup.run(7)


In [ ]:
# CELL 8 — Don tien trinh DATT cu
# Chi dung tien trinh cu da xac minh thuoc DATT; giu backend khoe.
startup.run(8)


In [ ]:
# CELL 9 — Khoi dong DATT bang CLI
# Khoi dong che do GPU va cac dich vu do CLI quan ly.
startup.run(9)


In [ ]:
# CELL 10 — Kiem tra backend va notification worker
# Chi PASS khi cac dich vu bat buoc deu khoe.
startup.run(10)


In [ ]:
# CELL 11 — Lay duong dan mo ung dung
# Mo URL Colab duoc in ra sau khi health check PASS.
startup.run(11)


In [ ]:
# CELL 12 — Smoke test API
# Chi doc API; khong bat camera hay tao du lieu nghiep vu.
startup.run(12)


In [ ]:
# CELL 13 — Tong ket trang thai san sang
# SYSTEM_READY=YES la startup PASS, khong phai Real Camera Acceptance.
startup.run(13)
